# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [173]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [174]:
df= pd.read_csv('data/AviationData.csv', encoding='cp1252')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

/var/folders/gc/qymlq6ns7yl2prvzhvfmqcrm0000gn/T/ipykernel_2903/2741149914.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df= pd.read_csv('data/AviationData.csv', encoding='cp1252')


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [175]:
df['Event.Date']= pd.to_datetime(df['Event.Date'])
df= df[(df['Amateur.Built'] == 'No') & (df['Event.Date'].dt.year >= 1983)]
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 76960 entries, 3600 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                76960 non-null  object        
 1   Investigation.Type      76960 non-null  object        
 2   Accident.Number         76960 non-null  object        
 3   Event.Date              76960 non-null  datetime64[ns]
 4   Location                76913 non-null  object        
 5   Country                 76750 non-null  object        
 6   Latitude                30167 non-null  object        
 7   Longitude               30161 non-null  object        
 8   Airport.Code            43375 non-null  object        
 9   Airport.Name            45285 non-null  object        
 10  Injury.Severity         75961 non-null  object        
 11  Aircraft.damage         73868 non-null  object        
 12  Aircraft.Category       25405 non-null  object  

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [176]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols]= df[injury_cols].fillna(0)
df['Estimated.Total.Passengers']= (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'] + df['Total.Minor.Injuries'] + df['Total.Uninjured'])
df= df[df['Estimated.Total.Passengers'] > 0]
df['Fatal/Serious.Injuries']= (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Estimated.Total.Passengers']
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 75701 entries, 3600 to 88888
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Event.Id                    75701 non-null  object        
 1   Investigation.Type          75701 non-null  object        
 2   Accident.Number             75701 non-null  object        
 3   Event.Date                  75701 non-null  datetime64[ns]
 4   Location                    75661 non-null  object        
 5   Country                     75495 non-null  object        
 6   Latitude                    29894 non-null  object        
 7   Longitude                   29888 non-null  object        
 8   Airport.Code                43119 non-null  object        
 9   Airport.Name                45009 non-null  object        
 10  Injury.Severity             75701 non-null  object        
 11  Aircraft.damage             73199 non-null  object      

I estimated the total number of passengers by summing Total.Fatal.Injuries, Total.Serious.Injuries, Total.Minor.Injuries, and Total.Uninjured. Since the dataset doesn't provide an explicit passenger count, this estimation may underestimate true occupancy if uninjured individuals were not recorded. Missing values in these columns were filled with 0, assuming unreported values indicate no recorded injuries. Rows with an estimated total of 0 were removed, as they likely represent incomplete data and would lead to invalid calculations. I then calculated the proportion of fatal or serious injuries relative to the estimated total passengers to measure accident severity.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [177]:
df= df[df['Aircraft.damage'].notna()]
df['Aircraft.Destroyed']= df['Aircraft.damage'] == 'Destroyed'
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 73199 entries, 3601 to 88886
Data columns (total 34 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Event.Id                    73199 non-null  object        
 1   Investigation.Type          73199 non-null  object        
 2   Accident.Number             73199 non-null  object        
 3   Event.Date                  73199 non-null  datetime64[ns]
 4   Location                    73163 non-null  object        
 5   Country                     73006 non-null  object        
 6   Latitude                    29043 non-null  object        
 7   Longitude                   29036 non-null  object        
 8   Airport.Code                42160 non-null  object        
 9   Airport.Name                44031 non-null  object        
 10  Injury.Severity             73199 non-null  object        
 11  Aircraft.damage             73199 non-null  object      

Since the Aircraft.damage column contained a small amount of missing values in comparison to the total number of rows in the dataset, I decided to remove the rows with a NaN value in order to  ensure reliability in the analysis. I then created a column (Aircraft.Destroyed) with binary variable, where accidents resulting in total aircraft destruction are labeled as True, and all other damage levels are labeled as False

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [178]:
df= df.dropna(subset=['Make'])
df['Make']= df['Make'].str.strip().str.title()

make_map= {
    'Airbus Industrie': 'Airbus',
    'Cirrus Design Corp.': 'Cirrus',
    'Cirrus Design Corp': 'Cirrus',
    'Robinson Helicopter': 'Robinson',
    'Robinson Helicopter Company': 'Robinson',
    'Dehavilland': 'De Havilland',
    'Air Tractor Inc': 'Air Tractor',
    'Aviat Aircraft Inc': 'Aviat',
    'Rockwell International': 'Rockwell'
}

df['Make']= df['Make'].replace(make_map)

make_counts= df['Make'].value_counts()
valid_makes= make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

For the Make column, missing values were first removed. The remaining entries were standardized by stripping leading and trailing whitespace and converting all values to title case. A mapping dictionary was then used to consolidate different representations of the same manufacturer into a single consistent label. Finally, the dataset was filtered to retain only manufacturers with 50 or more occurrences

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [179]:
df= df.dropna(subset = ['Model'])
df['Plane.Type'] = df['Make'] + ' ' + df['Model']

When inspecting the Make and Model columns, I found that model names were not globally unique across manufacturers. To fix this, I created a column that combined the Make with the Model in order to create a unique identifier for each plane type

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [180]:
df['Engine.Type'] = df['Engine.Type'].replace({
    'UNK': 'Unknown',
    'NONE': 'Unknown'
})
df['Weather.Condition'] = df['Weather.Condition'].replace({
    'Unk': 'UNK'
})
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'Air Race show': 'Air Race/show'
})
df = df[df['Number.of.Engines'] > 0]

I did minor cleaning for Engine.Type, Weather.Condition, and Purpose.of.flight by merging similar column names. For the Number.of.Engines column, I removed all entries with a value that was less than 1 since commercial and passenger planes can't have 0 engines. Since the directions said "You do not necessarily need to impute or drop NaNs here", I kept all of the NaN values for each column.

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
df= df.drop(columns=['Schedule', 'Air.carrier', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Aircraft.Category', 'FAR.Description'])


<class 'pandas.core.frame.DataFrame'>
Index: 62961 entries, 3601 to 88886
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Event.Id                    62961 non-null  object        
 1   Investigation.Type          62961 non-null  object        
 2   Accident.Number             62961 non-null  object        
 3   Event.Date                  62961 non-null  datetime64[ns]
 4   Location                    62944 non-null  object        
 5   Country                     62786 non-null  object        
 6   Injury.Severity             62961 non-null  object        
 7   Aircraft.damage             62961 non-null  object        
 8   Registration.Number         62916 non-null  object        
 9   Make                        62961 non-null  object        
 10  Model                       62961 non-null  object        
 11  Amateur.Built               62961 non-null  object      

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [182]:
df.to_csv('CleanedAviationData.csv')